# Edge浏览器
地址栏搜edge://flags/#enable-force-dark
- 在浏览器地址栏输入 chrome://flags 或 edge://flags，并按回车。<br>
- 找到 Force Dark Mode for Web Contents 选项，将其设置为 Enabled。<br>
- 重新启动浏览器。<br>
> 不再需要使用深色模式的浏览器扩展了

In [ ]:
#移动（而非复制）apk 文件并重命名，格式仍是：顶层目录_子目录1_子目录2_..._原文件名.apk
import os
import shutil

def move_and_rename_apks(root_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for folder, _, files in os.walk(root_dir):
        for file in files:
            if file.endswith(".apk"):
                full_path = os.path.join(folder, file)
                rel_path = os.path.relpath(full_path, root_dir)
                parts = os.path.normpath(rel_path).split(os.sep)
                new_name = "_".join(parts)
                new_path = os.path.join(output_dir, new_name)

                shutil.move(full_path, new_path)
                print(f"✅ 已移动并重命名: {new_name}")

# 使用示例：
move_and_rename_apks("你的根目录路径", "你想放置apk的目标目录")


In [ ]:
#把一个文件夹中的所有文件每 10 个一组，移动到新建的子文件夹中，子文件夹按数字命名：1/、2/、3/……

import os
import shutil

def group_files_by_ten(src_dir):
    files = [f for f in os.listdir(src_dir) if os.path.isfile(os.path.join(src_dir, f))]
    files.sort()  # 可选：按文件名排序
    group_size = 10

    for i in range(0, len(files), group_size):
        group = files[i:i + group_size]
        folder_name = str(i // group_size + 1)
        folder_path = os.path.join(src_dir, folder_name)
        os.makedirs(folder_path, exist_ok=True)

        for f in group:
            src_file = os.path.join(src_dir, f)
            dst_file = os.path.join(folder_path, f)
            shutil.move(src_file, dst_file)
            print(f"✅ 移动: {f} → {folder_name}/")

# 使用示例
group_files_by_ten("你的文件夹路径")


In [1]:
from pathlib import Path
import shutil

# 当前目录作为源和目标路径
source_path = Path(r"G:\YTDowmLoad\music")  # 替换为你的路径
group_size = 100  # 每组包含多少个文件夹

# 获取当前目录下所有文件夹（不包含文件）
folders = [f for f in sorted(source_path.iterdir()) if f.is_dir() and not f.name.startswith("Group")]

# 分组移动
for i in range(0, len(folders), group_size):
    group_index = (i // group_size) + 1
    group_folder = source_path / f"Group{group_index}"
    group_folder.mkdir(exist_ok=True)

    for folder in folders[i:i + group_size]:
        target = group_folder / folder.name
        shutil.move(str(folder), str(target))

print("分组完成。")


分组完成。


In [ ]:
from pathlib import Path
import shutil
import re

# 配置
base_path = Path(r"D:\MyFolders")  # 替换为你的路径
group_size = 100

# 1. 找出所有 GroupX 文件夹
group_folders = {
    int(match.group(1)): f
    for f in base_path.iterdir()
    if f.is_dir() and (match := re.fullmatch(r"Group(\d+)", f.name))
}
if group_folders:
    max_group_number = max(group_folders.keys())
    last_group = group_folders[max_group_number]
    last_group_count = len([f for f in last_group.iterdir() if f.is_dir()])
else:
    max_group_number = 0
    last_group = None
    last_group_count = 0

# 2. 找出“未分组”的文件夹
unassigned = [
    f for f in sorted(base_path.iterdir())
    if f.is_dir() and not re.fullmatch(r"Group\d+", f.name)
]

# 3. 准备分配：先补充 last_group，如果有的话
index = 0
if last_group and last_group_count < group_size:
    fill_count = min(group_size - last_group_count, len(unassigned))
    for folder in unassigned[:fill_count]:
        shutil.move(str(folder), str(last_group / folder.name))
    index = fill_count  # 从这里继续分组

# 4. 把剩下的继续新建分组
for i in range(index, len(unassigned), group_size):
    group_number = max_group_number + ((i - index) // group_size) + 1
    group_path = base_path / f"Group{group_number}"
    group_path.mkdir(exist_ok=True)

    for folder in unassigned[i:i + group_size]:
        shutil.move(str(folder), str(group_path / folder.name))

print("增量分组完成（补足+新建）。")


In [ ]:
from pathlib import Path
import shutil

# 当前目录路径
base_path = Path(r"D:\MyFolders")  # 替换为你的目录

# 查找 Group 开头的分组文件夹
group_folders = [f for f in base_path.iterdir() if f.is_dir() and f.name.startswith("Group")]

for group in group_folders:
    for subfolder in group.iterdir():
        if subfolder.is_dir():
            target = base_path / subfolder.name
            shutil.move(str(subfolder), str(target))
    # 移动完后删除空的 Group 文件夹
    group.rmdir()

print("还原完成。")
